In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from battimal.preprocessor.battery_data import EchemMeasurement, CellData, BatteryDataset

In [2]:
test_EchemMeasurement = EchemMeasurement(
    measurement_type='test',
    Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=[0.0, 1.0, 2.0],
    extractors={
        'max_voltage': lambda measurement: max(measurement.Voltage_V),
        'df_power': lambda measurement: pd.Series(np.array(measurement.Voltage_V) * np.array(measurement.Current_A)),
    })

test_EchemMeasurement

EchemMeasurement(measurement_type='test', Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=[0.0, 1.0, 2.0], extractors={'max_voltage': <function <lambda> at 0x118689800>, 'df_power': <function <lambda> at 0x118689440>}, extracts=None)

In [3]:
test_EchemMeasurement.run_extractors()
test_EchemMeasurement.extracts

{'max_voltage': 3.7,
 'df_power': 0    0.37
 1    0.72
 2    1.05
 dtype: float64}

In [ ]:
hdf5 = pd.HDFStore('test_data/test_EchemMeasurement.h5', mode='w')
test_EchemMeasurement.to_h5(hdf5)
keys = hdf5.keys()
hdf5.close()

keys

['/measurement']

In [ ]:
hdf5 = pd.HDFStore('test_data/test_EchemMeasurement.h5', mode='r')
test_EchemMeasurement_from_h5 = EchemMeasurement.from_h5('test', hdf5, 'measurement')
hdf5.close()

test_EchemMeasurement_from_h5

EchemMeasurement(measurement_type='test', Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=array([0., 1., 2.]), extractors=[], extracts={})

In [6]:
test_EchemMeasurement.run_extractors()
test_EchemMeasurement.extracts

{'max_voltage': 3.7,
 'df_power': 0    0.37
 1    0.72
 2    1.05
 dtype: float64}

In [7]:
test_echem_0 = EchemMeasurement(
    measurement_type='test A',
    Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=[0.0, 1.0, 2.0],
    extractors={
        'max_voltage': lambda measurement: max(measurement.Voltage_V),
        'df_power': lambda measurement: pd.Series(np.array(measurement.Voltage_V) * np.array(measurement.Current_A)),
    })
test_echem_1 = EchemMeasurement(
    measurement_type='test B',
    Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=[0.0, 1.0, 2.0],
    extractors={
        'min_voltage': lambda measurement: min(measurement.Voltage_V),
    })
test_echem_2 = EchemMeasurement(
    measurement_type='test A',
    Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=[0.0, 1.0, 2.0],
    extractors={
        'max_voltage': lambda measurement: max(measurement.Voltage_V),
        'df_power': lambda measurement: pd.Series(np.array(measurement.Voltage_V) * np.array(measurement.Current_A)),
    })

test_cell = CellData(
    cell_id='test_cell',
    measurement_extractors={
        'cumprod_max_current': lambda cell: np.cumprod([np.max(m.Current_A) for m in cell]).tolist(),
    },
    cell_extractors={
        'sum_time': lambda cell: np.sum([m.Time_s[-1] for m in cell]),
    }
)
test_cell.append(test_echem_0, start_time_s=0, cycles=1)
test_cell.append(test_echem_1, start_time_s=10, cycles=5)
test_cell.append(test_echem_2, start_time_s=110, cycles=1)
test_cell.run_extractors()

test_cell, test_cell.measurement_extracts, test_cell.cell_extracts

(Cell ID: test_cell, Total measurements: 3, Measurement types: {'test B': 1, 'test A': 2}
 'test A' 'test B' 'test A',
 {'min_voltage': [None, 3.5, None],
  'max_voltage': [3.7, None, 3.7],
  'df_power': [0    0.37
   1    0.72
   2    1.05
   dtype: float64,
   None,
   0    0.37
   1    0.72
   2    1.05
   dtype: float64],
  'cumprod_max_current': [0.3, 0.09, 0.027]},
 {'sum_time': np.float64(6.0)})

In [8]:
test_cell.scalar_measurement_extracts_to_df()

,min_voltage,max_voltage,cumprod_max_current
0,NaN,3.7,0.300
1,3.5,NaN,0.090
2,NaN,3.7,0.027


In [9]:
test_cell.measurement_metadata_to_df()

,measurement_type,measurement__class__,start_time_s,start_time_day,cycle,charge_throughput_Ah,repeat_num
0,test A,EchemMeasurement,0,0.000000,1,0.000111,1
1,test B,EchemMeasurement,10,0.000116,6,0.000222,1
2,test A,EchemMeasurement,110,0.001273,7,0.000333,2


In [ ]:
hdf5 = pd.HDFStore('test_data/test_CellData.h5', mode='w')
test_cell.to_h5(hdf5)

hdf5 = pd.HDFStore('test_data/test_CellData.h5', mode='r')
keys = hdf5.keys()
hdf5.close()

keys

['/test_cell/cell_extracts',
 '/test_cell/measurement_metadata',
 '/test_cell/measurements/measurement_0',
 '/test_cell/measurements/measurement_1',
 '/test_cell/measurements/measurement_2',
 '/test_cell/measurement_extracts/dataframe',
 '/test_cell/measurement_extracts/df_power/value_0',
 '/test_cell/measurement_extracts/df_power/value_2',
 '/test_cell/cell_metadata/__class__',
 '/test_cell/cell_metadata/cell_id']

In [ ]:
test_cell = CellData.from_h5('test_cell', pd.HDFStore('test_data/test_CellData.h5', mode='r'))
test_cell

Cell ID: test_cell, Total measurements: 3, Measurement types: {'test B': 1, 'test A': 2}
'test A' 'test B' 'test A'

In [12]:
test_cell.scalar_measurement_extracts_to_df()

,min_voltage,max_voltage,cumprod_max_current
0,NaN,3.7,0.300
1,3.5,NaN,0.090
2,NaN,3.7,0.027


In [13]:
test_cell.measurement_metadata_to_df()

,measurement_type,measurement__class__,start_time_s,start_time_day,cycle,charge_throughput_Ah,repeat_num
0,test A,EchemMeasurement,0,0.000000,1,0.000111,1
1,test B,EchemMeasurement,10,0.000116,6,0.000222,1
2,test A,EchemMeasurement,110,0.001273,7,0.000333,2


In [14]:
cells = []
for i in range(3):
    test_cell = CellData(
        cell_id=f'test_cell_{i}',
        measurement_extractors={
            'cumprod_max_current': lambda cell: np.cumprod([np.max(m.Current_A) for m in cell]).tolist(),
        },
        cell_extractors={
            'sum_time': lambda cell: np.sum([m.Time_s[-1] for m in cell]),
        }
    )
    test_cell.append(test_echem_0, start_time_s=0, cycles=1)
    test_cell.append(test_echem_1, start_time_s=10, cycles=5)
    test_cell.append(test_echem_2, start_time_s=110, cycles=1)

    cells.append(test_cell)

test_dataset = BatteryDataset(battery_data=cells, metadata={'dataset_name': 'test_dataset'})
test_dataset.run_extractors()

test_dataset.cell_extracts_and_metadata_to_df()

,cell_id,__class__,sum_time
0,test_cell_0,CellData,6.0
1,test_cell_1,CellData,6.0
2,test_cell_2,CellData,6.0


In [15]:
test_dataset[0].scalar_measurement_extracts_to_df()

,min_voltage,max_voltage,cumprod_max_current
0,NaN,3.7,0.300
1,3.5,NaN,0.090
2,NaN,3.7,0.027


In [ ]:
test_dataset.to_h5(pd.HDFStore('test_data/test_BatteryDataset.h5', mode='w'))

hdf5 = pd.HDFStore('test_data/test_BatteryDataset.h5', mode='r')
keys = hdf5.keys()
hdf5.close()

keys

['/metadata/dataset_name',
 '/battery_data/test_cell_2/cell_extracts',
 '/battery_data/test_cell_2/measurement_metadata',
 '/battery_data/test_cell_2/measurements/measurement_0',
 '/battery_data/test_cell_2/measurements/measurement_1',
 '/battery_data/test_cell_2/measurements/measurement_2',
 '/battery_data/test_cell_2/measurement_extracts/dataframe',
 '/battery_data/test_cell_2/measurement_extracts/df_power/value_0',
 '/battery_data/test_cell_2/measurement_extracts/df_power/value_2',
 '/battery_data/test_cell_2/cell_metadata/__class__',
 '/battery_data/test_cell_2/cell_metadata/cell_id',
 '/battery_data/test_cell_1/cell_extracts',
 '/battery_data/test_cell_1/measurement_metadata',
 '/battery_data/test_cell_1/measurements/measurement_0',
 '/battery_data/test_cell_1/measurements/measurement_1',
 '/battery_data/test_cell_1/measurements/measurement_2',
 '/battery_data/test_cell_1/measurement_extracts/dataframe',
 '/battery_data/test_cell_1/measurement_extracts/df_power/value_0',
 '/batter

In [ ]:
test_dataset = BatteryDataset.from_h5(BatteryDataset, pd.HDFStore('test_data/test_BatteryDataset.h5', mode='r'))
test_dataset.cell_extracts_and_metadata_to_df()

,cell_id,__class__,sum_time
0,test_cell_0,CellData,6.0
1,test_cell_1,CellData,6.0
2,test_cell_2,CellData,6.0


In [18]:
test_dataset[0].scalar_measurement_extracts_to_df()

,min_voltage,max_voltage,cumprod_max_current
0,NaN,3.7,0.300
1,3.5,NaN,0.090
2,NaN,3.7,0.027


In [19]:
test_dataset[0].measurement_metadata_to_df()

,measurement_type,measurement__class__,start_time_s,start_time_day,cycle,charge_throughput_Ah,repeat_num
0,test A,EchemMeasurement,0,0.000000,1,0.000111,1
1,test B,EchemMeasurement,10,0.000116,6,0.000222,1
2,test A,EchemMeasurement,110,0.001273,7,0.000333,2


In [20]:
test_dataset[0][0]

EchemMeasurement(measurement_type='test A', Current_A=[0.1, 0.2, 0.3], Voltage_V=[3.7, 3.6, 3.5], Step=[1, 2, 3], Time_s=array([0., 1., 2.]), extractors=[], extracts={})